# 第 1 周期末练习 —— 技术面试准备导师

## 练习目标（理念）

为展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个工具：接收技术问题，并返回解释。这是一个你在课程中也能自己用的工具！

本 notebook 进一步做成「按岗位定制」的技术面试教练：先选模型与目标岗位，再进入多轮对话循环。


## 技术面试准备导师

跨多个分析与 AI 岗位准备技术面试可能很吃力——每个岗位需要的技能、知识与深度都不同。本工具充当按岗位定制的技术教练。

### 功能
- 开始前先选择目标岗位（Data Scientist / ML Engineer 等）
- 充当该岗位的专属技术教练（靠不同的 system prompt）
- 回答任意技术问题，并尽量关联到所选岗位
- 支持云端 GPT 与本地 Ollama（OpenAI 兼容 `/v1`）两条后端

### 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `client.chat.completions.create(...)` |
| system / user / assistant | 多轮 `messages` 列表模拟「记忆」 |
| Ollama 兼容接口 | `base_url="http://localhost:11434/v1"` |
| 角色化 prompt | `role_prompts` 字典按岗位切换人设 |

### 怎么跑

1. 先确认本机 Ollama 已启动；需要本地模型时执行 `ollama pull qwen2.5-coder`
2. `.env` 准备好 `OPENAI_API_KEY`（走 GPT 时）
3. 从上到下运行；最后一格会进入交互式 `input()` 对话，输入 `quit` 退出


In [ ]:
# ========== 探活：确认本机 Ollama 服务是否在跑 ==========
# 导入 requests：发 HTTP GET 探测本地 Ollama 默认端口
import requests
# 访问 http://localhost:11434 ；若服务未启动会报连接错误
# .content：看响应原始字节（Ollama 根路径通常会回一段简短说明）
requests.get("http://localhost:11434").content


In [ ]:
# ========== 拉取本地模型：把 qwen2.5-coder 下载到 Ollama ==========
# Jupyter 里 ! 开头表示执行 shell 命令（不是纯 Python）
!ollama pull qwen2.5-coder


In [ ]:
# ========== 安装依赖：openai SDK 与 python-dotenv ==========
# 用 uv pip 安装；若环境已装过可跳过本格
!uv pip install openai python-dotenv


In [ ]:
# ========== 导入：环境变量相关 ==========
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# 导入标准库 os：需要时可读 os.getenv（本练习密钥主要由 OpenAI() 默认读取）
import os


In [ ]:
# ========== 导入：API 客户端与笔记本展示 ==========
# OpenAI 客户端：既可连云端，也可通过 base_url 连 Ollama 的 OpenAI 兼容端点
from openai import OpenAI
# Markdown + display：把导师回答渲染成排版后的 Markdown
from IPython.display import Markdown, display


In [ ]:
# ========== 常量：模型名字集中写在一处 ==========

# 云端 OpenAI 模型：较快；需有效 API Key
MODEL_GPT = 'gpt-4.1-mini'
# 本地 Ollama 模型名：需事先 ollama pull；字符串须与本机已安装名一致
MODEL_OLLAMA = 'qwen2.5-coder'


In [ ]:
# ========== 环境 + 双客户端：云端 GPT 与本地 Ollama 各一个 ==========

# 加载 .env；override=True 允许用文件覆盖已有环境变量
load_dotenv(override=True)

# 默认 OpenAI 云端客户端（读 OPENAI_API_KEY）
openai_client = OpenAI()

# Ollama：走 OpenAI 兼容的 /v1；api_key 占位即可（本地通常不校验）
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)


## Prompt 设计说明

本工具**主要靠 system prompt**——对话循环里用户问题直接作为 `role: user` 追加进 `messages`，不必再单独拼一套「固定 user 模板」。

### 通用指令 `general_instruction`
拼到每个岗位 prompt 末尾，保证：
- 任何技术问题都答，不拒绝、不随便甩锅
- 相关时把答案连回所选岗位，并给实用例子
- 有帮助时尽量给代码示例

### 岗位专用 system prompts
`role_prompts` 字典里 5 套人设——运行时按用户选择动态切换导师专长与侧重点：
- 每条定义岗位、聚焦技能、面试辅导策略
- 与 `general_instruction` 拼接后，构成完整行为约束


In [ ]:
# ========== Prompt：通用指令 + 按岗位区分的 system prompts 字典 ==========
# 下面多段英文是发给模型的指令，保留英文以免改变回答风格/行为
general_instruction = """

You should answer ANY technical question the person asks — 
never refuse or redirect. If they ask about topics outside their 
role, answer fully and helpfully. When relevant, connect it back 
to how it applies to their specific role with a practical example.

Always be encouraging, clear and provide code examples where helpful.
"""

# key "1".."5" 对应后面 start_tutor() 里让用户输入的岗位编号
role_prompts = {
    "1": """You are a technical coach and an expert at answering 
questions for a Data Scientist role. Guide this person technically 
to ace their interview by explaining concepts clearly, providing 
relevant code examples, and focusing on skills that matter most 
for Data Science roles such as statistics, ML algorithms, 
Python, and model evaluation.""" + general_instruction,

    "2": """You are a technical coach and an expert at answering 
questions for a Machine Learning Engineer role. Guide this person 
technically to ace their interview by focusing on ML systems design, 
model deployment, MLOps, Python, and production ML concepts.""" + general_instruction,

    "3": """You are a technical coach and an expert at answering 
questions for a Business Analyst role. Guide this person technically 
to ace their interview by focusing on SQL, data visualization, 
requirements gathering, stakeholder communication and analytical 
thinking.""" + general_instruction,

    "4": """You are a technical coach and an expert at answering 
questions for a Data Analyst role. Guide this person technically 
to ace their interview by focusing on SQL, Excel, Python basics, 
data visualization, and statistical analysis.""" + general_instruction,

    "5": """You are a technical coach and an expert at answering 
questions for an AI/LLM Engineer role. Guide this person technically 
to ace their interview by focusing on LLMs, prompt engineering, 
RAG, fine tuning, agents, and Python.""" + general_instruction,
}


## `chat()`：多轮对话循环

`chat()` 负责整段会话：
- 用所选岗位的 system prompt 初始化 `messages`
- `while True` 持续接收用户问题
- 每轮把问答追加进 `messages`，保留完整历史
- 每次 API 调用都带上整段历史——在无状态 API 之上「模拟记忆」
- 用 Markdown 展示回答
- 用户输入 `quit` 时优雅退出


In [ ]:
# ========== chat：带历史的多轮问答循环 ==========

def chat(model, client, role_choice):
    # 1) 初始化 messages：第一条固定是该岗位的 system prompt
    messages = [
        {"role": "system", "content": role_prompts[role_choice]}
    ]


    # 2) 无限循环：直到用户输入 quit
    while True:
        # 读取用户输入（阻塞）；提示字符串保持英文原样
        user_input = input("You: ")

        # 退出条件：恰好输入 quit（区分大小写，与原逻辑一致）
        if user_input == "quit":
            print("Good luck with your interview!")
            break

        # 把本轮用户话追加进历史
        messages.append({"role": "user", "content": user_input})
        # 回显用户问题，并打印 Tutor 标题
        print(f"\nYou: {user_input}")
        print("\nTutor:")

        # 调用 Chat Completions：messages 带完整对话，模拟多轮记忆
        response = client.chat.completions.create(
            model=model,
            messages=messages, # entire conversation every time!
        )

        # 取出助手文本并渲染为 Markdown
        answer = response.choices[0].message.content
        display(Markdown(answer))

        # 把助手回答也写回历史，供下一轮上下文使用
        messages.append({"role": "assistant", "content": answer})


## `start_tutor()`：会话前的设置向导

`start_tutor()` 在真正聊天前完成准备：
- 打印欢迎语与可用选项
- 读模型选择 → 绑定对应的 `client` 与 `model`
- 读岗位选择 → 决定用 `role_prompts` 里哪一条
- 每步校验：输入 `quit` 则退出
- 最后把模型、客户端、岗位传给 `chat()` 开始会话


In [ ]:
# ========== start_tutor：选模型、选岗位，再进入 chat() ==========

def start_tutor():
    # 欢迎语与退出提示（给人看的 print 文案保持英文原样，避免改交互体验）
    print("Welcome to the Technical Interview Tutor!")
    print("Ask me any technical question and I'll help you prepare for your interview!")
    print("Type 'quit' to end the conversation.")

    # 先选后端模型：1 = 云端 GPT，2 = 本地 Ollama
    print("\nChoose your model:")
    print("1. GPT-4.1-mini (OpenAI - fast)")
    print("2. Qwen2.5-coder (Ollama - free, slower)")
    model_choice = input("Enter 1 or 2: ")

    # 允许在选模型阶段用 quit 退出
    if model_choice.lower() == "quit":
        print("Goodbye!")
        return

    # 按选择绑定 client 与 model 常量
    if model_choice == "1":
        client = openai_client
        model = MODEL_GPT
        print(f"\nUsing GPT-4.1-mini!")
    else:
        # 非 "1" 都走 Ollama（与原逻辑一致，未做严格校验）
        client = ollama_client
        model = MODEL_OLLAMA
        print(f"\nUsing Qwen2.5-coder!")

    # 再选目标岗位：编号对应 role_prompts 的 key
    print("\nWhich role are you preparing for?")
    print("1. Data Scientist")
    print("2. Machine Learning Engineer")
    print("3. Business Analyst")
    print("4. Data Analyst")
    print("5. AI/LLM Engineer")
    role_choice = input("Enter 1-5: ")
    if role_choice.lower() == "quit":
        print("Goodbye!")
        return

    # 编号 → 可读岗位名，仅用于打印确认
    role_names = {
    "1": "Data Scientist",
    "2": "Machine Learning Engineer",
    "3": "Business Analyst",
    "4": "Data Analyst",
    "5": "AI/LLM Engineer"
    }
    print(f"\nPreparing you for: {role_names[role_choice]} role!")
    print(f"Let's get started!\n")  

    # 进入多轮对话；传入已选好的 model / client / role_choice
    chat(model, client, role_choice)

# 运行本格即启动导师（交互式；在无 input 的环境会阻塞）
start_tutor()
